# Titanic: построение и оценка моделей 
Цель: подготовить признаки на основе выводов из EDA, обучить базовые модели и зафиксировать стартовую метрику (`Accuracy`) на кросс-валидации.

In [1]:
import numpy as np 
import pandas as pd 
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.compose import ColumnTransformer 
from sklearn.pipeline import Pipeline 
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

train = pd.read_csv("../data/train.csv")
test = pd.read_csv("../data/test.csv")

def add_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    df["Title"] = df["Name"].str.extract(r",\s*([^\.]+)\.", expand=False)
    rare_titles = df["Title"].value_counts()[df["Title"].value_counts() < 10].index
    df["Title"] = df["Title"].replace(rare_titles, "Rare")
    df["Title"] = df["Title"].replace({"Mile" : "Miss", "Ms": "Miss", "Mme" : "Mrs"})

    df["FamilySize"] = df["SibSp"] + df["Parch"] + 1
    df["IsAlone"] = (df["FamilySize"] == 1).astype(int)


    df["HasCabin"] = df["Cabin"].notnull().astype(int)  

    return df

train_feat = add_features(train)
test_feat = add_features(test)

## Обработка пропусков и создание взаимодействий
- Пропуски в `Age` заполняем медианным возрастом соответствующего обращения (`Title`), считая медианы строго по `train` во избежание утечки данных.
- Добавляем бинарный признак `IsChild` (возраст `<= 6` лет) и комбинированный признак `Sex_Pclass` для учета разного влияния класса каюты на мужчин и женщин.

In [2]:
title_age_medians = train_feat.groupby("Title")["Age"].median()
for title, median_age in title_age_medians.items():
    train_feat.loc[(train_feat["Age"].isnull()) & (train_feat["Title"] == title), "Age"] = median_age
    test_feat.loc[(test_feat["Age"].isnull()) & (test_feat["Title"] == title), "Age"] = median_age
    
global_age_median = train_feat["Age"].median()
train_feat["Age"] = train_feat["Age"].fillna(global_age_median)
test_feat["Age"] = test_feat["Age"].fillna(global_age_median)


train_feat["IsChild"] = (train_feat["Age"] <= 6).astype(int)
test_feat["IsChild"] = (test_feat["Age"] <= 6).astype(int)

train_feat["Sex_Pclass"] = train_feat["Sex"] + "_" + train_feat["Pclass"].astype(str)
test_feat["Sex_Pclass"] = test_feat["Sex"] + "_" + test_feat["Pclass"].astype(str)

num_features = ["Age", "Fare", "FamilySize", "IsAlone", "HasCabin", "IsChild"]
cat_features = ["Pclass", "Sex", "Embarked", "Title", "Sex_Pclass"]

X = train_feat[num_features + cat_features]
y = train_feat["Survived"]
X_test = test_feat[num_features + cat_features]


In [3]:
cat_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("ohe", OneHotEncoder(handle_unknown="ignore"))
])

num_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

preprocessor = ColumnTransformer(transformers=[
    ("num", num_transformer, num_features),
    ("cat", cat_transformer, cat_features)
])

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

models = {
    "LogisticRegression": LogisticRegression(max_iter= 1000, random_state= 42),
    "RandomForest": RandomForestClassifier(n_estimators= 200, max_depth= 6, random_state=42)
}

for name, model in models.items():
    clf = Pipeline(steps=[("preprocessor", preprocessor), ("model", model)])
    scores = cross_val_score(clf, X, y, cv=cv, scoring="accuracy")
    print(f"{name}: {scores.mean():.4f} +- {scores.std():.4f}")

LogisticRegression: 0.8373 +- 0.0118


RandomForest: 0.8339 +- 0.0129



## Результаты бейзлайна
- `LogisticRegression`: **0.8373 +- 0.0118**
- `RandomForest`: **0.8339 +- 0.0129**

Обе модели уверенно обошли простое правило по полу (0.787). Линейная модель показала лучший результат за счет сгенерированных взаимодействий (`Sex_Pclass`, `Title`, `IsChild`).

In [11]:
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import GridSearchCV

gb_clf = Pipeline(steps=[
    ("preprocessor", preprocessor), 
    ("model", GradientBoostingClassifier(n_estimators=150, learning_rate=0.05, max_depth=3, random_state=42))
])

gb_scores = cross_val_score(gb_clf, X, y, cv=cv, scoring="accuracy")
print(f"GradientBoosting: {gb_scores.mean():.4f} +- {gb_scores.std():.4f}")

param_grid = {"model__C": [0.01, 0.05, 0.1, 0.5, 2.0, 5.0, 10.0]}
lr_pipe = Pipeline(steps=[
    ("preprocessor", preprocessor), 
    ("model", LogisticRegression(max_iter=1000, random_state=42))
])

grid = GridSearchCV(lr_pipe, param_grid, cv=cv, scoring="accuracy", n_jobs=-1)
grid.fit(X, y)

print(f"Best LogisticRegression (C={grid.best_params_['model__C']}) : {grid.best_score_:.4f}")

GradientBoosting: 0.8429 +- 0.0214


Best LogisticRegression (C=0.1) : 0.8373


In [14]:
if gb_scores.mean() > grid.best_score_:
    best_model = gb_clf 
    best_name = "GradientBoosting"
else: 
    best_model = grid.best_estimator_
    best_name = f"LogisticRegression (C={grid.best_params_['model__C']})"

best_model.fit(X, y)

predictions = best_model.predict(X_test)

submission = pd.DataFrame({
    "PassengerId": test["PassengerId"],
    "Survived": predictions.astype(int)
})

submission.to_csv("../data/submission.csv", index=False)
print(f"Использована модель: {best_name}")
print(f"Файл .,/data/submission.csv сохранен ({len(submission)} строк). Доля выживших: {submission['Survived'].mean():.3f}")
submission.head()



Использована модель: GradientBoosting
Файл .,/data/submission.csv сохранен (418 строк). Доля выживших: 0.342


,PassengerId,Survived
0,892,0
1,893,0
2,894,0
3,895,0
4,896,1


## Итоги 
- Лучшей моделью на 5-fold кросс-валидации стал `GradientBoostingClassifier`, обошедший базовую логистическую регрессию и случайный лес.
- Сформирован файл предсказаний `data/submission.csv` для тестовой выборки (418 пассажиров, доля выживших: `0.342`).
- Скор на kaggle - 0.77272